<a href="https://colab.research.google.com/github/MauricioLlugdar/CoffeeShop-Sales/blob/main/01_CoffeeShopSales_Analysis_Exploration_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coffee Shop Sales (Dirty Dataset) - Dataset Analysis and Exploration

## Plan de trabajo:

**Etapa inicial:** sería ver de analizar bien la data, hacer una limpieza y feature engineering.

**Accion**: Luego de la etapa inicial, el plan es predecir la cantidad vendida de la semana siguiente. Para esto deberemos crear las features necesarias para la predicción.

**Extra**: Lo ideal sería analizar cuanta cantidad se va a vender de cada articulo, para de esta manera saber cuanto stock se necesitará.

In [163]:
!pip install kagglehub

In [164]:
import kagglehub
import pandas as pd
import numpy as np

In [165]:
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
Path to dataset files: /kaggle/input/cafe-sales-dirty-data-for-cleaning-training


In [166]:
df_raw = pd.read_csv(path + "/dirty_cafe_sales.csv")
df = df_raw.copy()
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Analisis general del dataset

In [167]:
df.shape

(10000, 8)

In [168]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

In [169]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [170]:
df.isna().mean().sort_values(ascending=False)

,0
Location,0.3265
Payment Method,0.2579
Item,0.0333
Price Per Unit,0.0179
Total Spent,0.0173
Transaction Date,0.0159
Quantity,0.0138
Transaction ID,0.0000


In [171]:
print(df.duplicated().sum())


0


### Analisis de cada feature

In [172]:
print(df.nunique().sort_values())

Location                4
Payment Method          5
Quantity                7
Price Per Unit          8
Item                   10
Total Spent            19
Transaction Date      367
Transaction ID      10000
dtype: int64


In [173]:
exclude_cols = ['Transaction ID', 'Transaction Date']

for col in df.columns:
    if col not in exclude_cols:
        unique_vals = df[col].unique()
        print(f"{col} ({len(unique_vals)} valores únicos):\n {unique_vals}\n")
    else:
        unique_vals = df[col].unique()[:5]
        print(f"{col} (Mostrando 5 de {len(df[col].unique())}):\n {unique_vals}...\n")

Transaction ID (Mostrando 5 de 10000):
 ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411']...

Item (11 valores únicos):
 ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

Quantity (8 valores únicos):
 ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

Price Per Unit (9 valores únicos):
 ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

Total Spent (20 valores únicos):
 ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

Payment Method (6 valores únicos):
 ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

Location (5 valores únicos):
 ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

Transaction Date (Mostrando 5 de 368):
 ['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11']...



In [174]:
df["Transaction ID"].nunique()


10000

In [175]:
df["Transaction ID"].duplicated().sum()

np.int64(0)

Como no hay transacciones duplicadas, ni errores de carga, entonces elimino esta feature, ya que no va a ser de ayuda en la predicción

In [176]:
df = df.drop(columns=["Transaction ID"])

Lo mejor que se puede hacer con las variables categoricas es aplicarles One Hot Encoding para que el modelo pueda predecir gracias a las mismas. Las variables categoricas que poseemos están dadas por las features:


*   Item
*   Payment Method
*   Location



Primero unifico los errores y los unknowns con nan. También limpio los strings y valores numericos

In [177]:
df = df.replace(["UNKNOWN", "ERROR"], np.nan)

for col in ["Item", "Payment Method", "Location"]:
    df[col] = df[col].astype("string").str.strip()

for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")


Veamos el resultado de esta limpieza

In [178]:
for col in df.columns:
    if col not in exclude_cols:
        unique_vals = df[col].unique()
        print(f"{col} ({len(unique_vals)} valores únicos):\n {unique_vals}\n")
    else:
        unique_vals = df[col].unique()[:5]
        print(f"{col} (Mostrando 5 de {len(df[col].unique())}):\n {unique_vals}...\n")

Item (9 valores únicos):
 <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',       <NA>,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: string

Quantity (6 valores únicos):
 [ 2.  4.  5.  3.  1. nan]

Price Per Unit (7 valores únicos):
 [2.  3.  1.  5.  4.  1.5 nan]

Total Spent (18 valores únicos):
 [ 4.  12.   nan 10.  20.   9.  16.  15.  25.   8.   5.   3.   6.   2.
  1.   7.5  4.5  1.5]

Payment Method (4 valores únicos):
 <StringArray>
['Credit Card', 'Cash', <NA>, 'Digital Wallet']
Length: 4, dtype: string

Location (3 valores únicos):
 <StringArray>
['Takeaway', 'In-store', <NA>]
Length: 3, dtype: string

Transaction Date (Mostrando 5 de 366):
 <DatetimeArray>
['2023-09-08 00:00:00', '2023-05-16 00:00:00', '2023-07-19 00:00:00',
 '2023-04-27 00:00:00', '2023-06-11 00:00:00']
Length: 5, dtype: datetime64[ns]...



Veamos como está la cabeza del df hasta ahora

In [179]:
df.head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
3,Salad,2.0,5.0,10.0,<NA>,<NA>,2023-04-27
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Analisis de Variables Numéricas

Como se puede ver el valor de total spent en la fila 2 NaN puede ser reemplazado facilmente por Quantity * Price per Unit para tener completitud en los datos.

In [180]:
df.loc[df["Total Spent"] != df["Quantity"] * df["Price Per Unit"]]

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
20,Smoothie,NaN,4.0,20.0,Cash,In-store,2023-04-04
25,Smoothie,3.0,4.0,NaN,<NA>,<NA>,2023-12-13
31,<NA>,2.0,1.0,NaN,Credit Card,<NA>,2023-11-06
42,Tea,2.0,1.5,NaN,<NA>,Takeaway,2023-01-10
...,...,...,...,...,...,...,...
9984,Smoothie,NaN,4.0,4.0,Cash,Takeaway,2023-07-27
9988,Cake,5.0,3.0,NaN,<NA>,<NA>,NaT
9993,Smoothie,2.0,4.0,NaN,Cash,<NA>,2023-10-20
9996,<NA>,3.0,NaN,3.0,Digital Wallet,<NA>,2023-06-02


Como vemos, tenemos que rellenar datos donde Total Spent es un NaN y tenemos la cantidad y el precio, tambien debemos rellenar los variables donde podemos obtener su valor infiriendo del resto

In [181]:
# 1. Creamos una máscara para identificar dónde podemos calcular el total
# (Necesitamos que Quantity y Price Per Unit no sean nulos)
mask_puedo_calcular = df['Quantity'].notna() & df['Price Per Unit'].notna()

# 2. Dentro de esas filas, buscamos las que están mal (valor distinto al producto o es NaN)
mask_error = (df['Total Spent'] != (df['Quantity'] * df['Price Per Unit']))

# Combinamos ambas: actualizamos solo si tenemos los datos y hay un error/nulo
condicion_final = mask_puedo_calcular & mask_error

df.loc[condicion_final, 'Total Spent'] = df['Quantity'] * df['Price Per Unit']

print(f"Se corrigieron {condicion_final.sum()} filas.")

Se corrigieron 462 filas.


In [182]:
df.head()

,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,Salad,2.0,5.0,10.0,<NA>,<NA>,2023-04-27
4,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


Pueden haber casos donde Quantity o Price Per Unit se puedan inferir, por lo que intentaremos hacerlo

Caso donde Quantity es un NaN y se puede inferir

In [183]:
mask_quantity = df['Quantity'].isna() & df["Total Spent"].notna() & df['Price Per Unit'].notna()

print(df.loc[mask_quantity].head())

         Item  Quantity  Price Per Unit  Total Spent  Payment Method  \
20   Smoothie       NaN             4.0         20.0            Cash   
55     Cookie       NaN             1.0          2.0     Credit Card   
57       Cake       NaN             3.0          3.0  Digital Wallet   
66      Juice       NaN             3.0          6.0            Cash   
117     Juice       NaN             3.0          9.0  Digital Wallet   

     Location Transaction Date  
20   In-store       2023-04-04  
55   Takeaway       2023-03-19  
57   In-store       2023-04-19  
66       <NA>       2023-03-30  
117      <NA>       2023-01-10  


Como podemos ver se puede inferir el valor de Quantity

In [184]:
df.loc[mask_quantity, "Quantity"] = df['Total Spent'] * (1/df['Price Per Unit'])

mask_quantity = df['Quantity'].isna() & df["Total Spent"].notna() & df['Price Per Unit'].notna()
print(mask_quantity.sum())

0


Como vemos ya no quedan valores de Quantity que se puedan corregir.

Ahora corrijamos el Price Per Unit

In [185]:
mask_ppp = df['Price Per Unit'].isna() & df["Total Spent"].notna() & df['Quantity'].notna()

print(df.loc[mask_ppp].head())

      Item  Quantity  Price Per Unit  Total Spent Payment Method  Location  \
56    Cake       5.0             NaN         15.0           <NA>  Takeaway   
68   Salad       2.0             NaN         10.0           <NA>  In-store   
85     Tea       3.0             NaN          4.5           Cash      <NA>   
104  Juice       2.0             NaN          6.0           <NA>      <NA>   
118   <NA>       5.0             NaN         15.0           <NA>  In-store   

    Transaction Date  
56        2023-06-27  
68        2023-10-27  
85        2023-10-29  
104              NaT  
118       2023-02-06  


In [186]:
df.loc[mask_ppp, "Price Per Unit"] = df['Total Spent'] * (1/df['Quantity'])

mask_ppp = df['Price Per Unit'].isna() & df["Total Spent"].notna() & df['Quantity'].notna()
print(mask_ppp.sum())

0


Veamos ahora los datos faltantes, para comparar con el inicio del analisis del dataset

In [187]:
df.isna().mean().sort_values(ascending=False)

,0
Location,0.3961
Payment Method,0.3178
Item,0.0969
Transaction Date,0.0460
Total Spent,0.0040
Price Per Unit,0.0038
Quantity,0.0038


Como vemos hay un decremento importante de filas donde faltan los valores de Quantity, Price Per Unit y Total Spent

### Analisis de Variables Categoricas

Como vemos faltan un 39,61% de los valores de Location y un 31,78% de los valores de Payment Method, lo que nos puede estar diciendo algo, por esta razón no vamos a promediar o rellenar con la moda, ni tampoco descartar las filas en las que falta. Vamos a utilizar otra categoría para referirnos a estos casos.

Analizando la otra variable Item, dada su cantidad y precio unitario, se podría inferir que Item es, por lo que se podrían recuperar varios Items y tener menos valores NaN.

Y por último, la feature transaction Date nos es escencial para nuestra predicción temporal, por lo que sin esta, no podemos predecir el consumo de la semana siguiente.
